# NB01 — Auditoría y Carga de Datos
## IEEE-CIS Fraud Detection | Data Analyst Track

**Objetivo:** Entender la estructura del dataset, calidad de datos, distribución del target y sentar las bases para el análisis exploratorio.

**Preguntas que responde este notebook:**
- ¿Cuántas transacciones tenemos y qué tan grande es el problema de fraude?
- ¿Qué tan completos están los datos? ¿Dónde están los huecos críticos?
- ¿Qué variables tienen alta tasa de nulos y qué implica eso?
- ¿Cuál es el perfil básico de una transacción fraudulenta vs. legítima?

---


## 1. Imports y Configuración

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')

# Configuración visual
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (12, 5)

# Rutas
DATA_DIR = os.path.dirname(os.getcwd()) if 'notebooks' in os.getcwd() else '.'
EXPORT_DIR = os.path.join(DATA_DIR, 'data', 'processed')

TX_PATH = os.path.join(DATA_DIR, 'data', 'raw', 'train_transaction.csv')
ID_PATH = os.path.join(DATA_DIR, 'data', 'raw', 'train_identity.csv')

print(f"Ruta transacciones: {TX_PATH}")
print(f"Ruta identidad:     {ID_PATH}")


## 2. Carga de Datos

In [ ]:
print("Cargando train_transaction.csv (683 MB)...")
tx = pd.read_csv(TX_PATH)
print(f"  ✓ {tx.shape[0]:,} filas × {tx.shape[1]} columnas")

print("Cargando train_identity.csv...")
id_ = pd.read_csv(ID_PATH)
print(f"  ✓ {id_.shape[0]:,} filas × {id_.shape[1]} columnas")

# JOIN: transacciones → identidad (LEFT JOIN)
df = tx.merge(id_, on='TransactionID', how='left')
print(f"\nDataset unificado: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"Transacciones CON datos de identidad: {id_.shape[0]:,} ({id_.shape[0]/tx.shape[0]*100:.1f}%)")
print(f"Transacciones SIN datos de identidad: {tx.shape[0]-id_.shape[0]:,} ({(tx.shape[0]-id_.shape[0])/tx.shape[0]*100:.1f}%)")


### Resultado de la Carga de Datos

El dataset combina `train_transaction.csv` con `train_identity.csv` mediante un **LEFT JOIN** sobre `TransactionID`.

| Dimensión | Valor |
|:---|---:|
| Total de transacciones | **590,540** |
| Total de variables | **434** (394 transaccionales + 40 de identidad) |
| Transacciones CON datos de identidad | 144,233 **(24.4%)** |
| Transacciones SIN datos de identidad | 446,307 **(75.6%)** |
| Período cubierto | ~184 días (~6 meses) |
| Tamaño en disco (raw) | ~683 MB (transaction) + ~23 MB (identity) |

> **Insight estructural:** La ausencia de datos de identidad no es un error de ETL — es un rasgo del e-commerce. La mayoría de compras online no pasan por autenticación explícita. En NB04, la *presencia o ausencia* de `id_XX` será en sí misma una señal predictiva importante para el modelo.

## 3. Distribución del Target: ¿Cuán grande es el problema?

In [ ]:
target_counts = df['isFraud'].value_counts()
fraud_rate = df['isFraud'].mean()
fraud_n = target_counts[1]
legit_n = target_counts[0]
total_amt = df['TransactionAmt'].sum()
fraud_amt = df.loc[df['isFraud']==1, 'TransactionAmt'].sum()

print("=" * 55)
print("  RESUMEN EJECUTIVO — DATASET")
print("=" * 55)
print(f"  Total de transacciones:        {df.shape[0]:>12,}")
print(f"  Transacciones legítimas:       {legit_n:>12,}  ({100-fraud_rate*100:.1f}%)")
print(f"  Transacciones fraudulentas:    {fraud_n:>12,}  ({fraud_rate*100:.2f}%)")
print(f"  Período cubierto:              6 meses (dataset)")
print(f"  Monto total transaccionado:    ${total_amt:>12,.0f}")
print(f"  Monto en riesgo (fraude):      ${fraud_amt:>12,.0f}  ({fraud_amt/total_amt*100:.1f}% del total)")
print("=" * 55)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Gráfico 1: Conteo
colors = ['#4CAF50', '#E53935']
axes[0].bar(['Legítima', 'Fraude'], [legit_n, fraud_n], color=colors, edgecolor='white', width=0.5)
axes[0].set_title('Distribución de Transacciones', fontweight='bold', fontsize=13)
axes[0].set_ylabel('Cantidad')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
for i, v in enumerate([legit_n, fraud_n]):
    axes[0].text(i, v + 3000, f'{v:,}\n({v/df.shape[0]*100:.1f}%)', ha='center', fontsize=11, fontweight='bold')

# Gráfico 2: Monto en riesgo
axes[1].bar(['Legítima', 'Fraude'], [total_amt - fraud_amt, fraud_amt], color=colors, edgecolor='white', width=0.5)
axes[1].set_title('Monto Transaccionado ($USD)', fontweight='bold', fontsize=13)
axes[1].set_ylabel('Monto total ($)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.0f}M'))
for i, v in enumerate([total_amt - fraud_amt, fraud_amt]):
    axes[1].text(i, v + total_amt * 0.005, f'${v/1e6:.1f}M\n({v/total_amt*100:.1f}%)', ha='center', fontsize=11, fontweight='bold')

plt.suptitle('IEEE-CIS Fraud Detection — Magnitud del Problema', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'assets', 'nb01_target_distribution.png'), bbox_inches='tight')
plt.show()
print("\n→ Gráfico guardado.")


### Distribución del Target — El Desafío del Desbalance Severo

| Clase | Conteo | Porcentaje | Implicación |
|:---|---:|---:|:---|
| Legítima (`isFraud = 0`) | 569,877 | **96.5%** | Clase mayoritaria |
| Fraude (`isFraud = 1`) | **20,663** | **3.5%** | "Aguja en el pajar" |

Este desbalance define la estrategia técnica y de evaluación del proyecto:

- **Métrica incorrecta:** Un clasificador que predice "0" siempre alcanza 96.5% de *accuracy* — pero detecta **0 fraudes**. La *accuracy* es una métrica trampa en este problema.  
- **Métrica correcta:** **AUC-ROC** mide la capacidad discriminativa real. Se complementa con la curva de ganancia acumulada en NB04.  
- **Visualizaciones:** Siempre se usarán **tasas de fraude (%)**, nunca conteos absolutos, para que el volumen no enmascare el riesgo relativo.  
- **Modelado:** El clasificador en NB04 usará `class_weight='balanced'` para compensar el desbalance sin recurrir a oversampling.

## 4. Auditoría de Calidad de Datos

In [ ]:
# Análisis de nulos por grupos de variables
def null_audit(dataframe, group_name, cols):
    """Calcula tasa de nulos para un grupo de columnas."""
    subset = dataframe[cols]
    null_pct = subset.isnull().mean() * 100
    return pd.DataFrame({
        'grupo': group_name,
        'variable': null_pct.index,
        'pct_nulos': null_pct.values
    })

# Agrupar variables según el diccionario
grupos = {
    'Identificadores/Target': ['TransactionID', 'isFraud', 'TransactionDT', 'TransactionAmt', 'ProductCD'],
    'Tarjeta (card1-6)': [c for c in df.columns if c.startswith('card')],
    'Dirección (addr/dist)': [c for c in df.columns if c.startswith('addr') or c.startswith('dist')],
    'Email': ['P_emaildomain', 'R_emaildomain'],
    'Conteos C (C1-C14)': [f'C{i}' for i in range(1, 15) if f'C{i}' in df.columns],
    'Timedeltas D (D1-D15)': [f'D{i}' for i in range(1, 16) if f'D{i}' in df.columns],
    'Match M (M1-M9)': [f'M{i}' for i in range(1, 10) if f'M{i}' in df.columns],
    'Vesta V (V1-V339)': [c for c in df.columns if c.startswith('V')],
    'Identidad id_XX': [c for c in df.columns if c.startswith('id_')],
    'Dispositivo': ['DeviceType', 'DeviceInfo'],
}

audit_rows = []
for grupo, cols in grupos.items():
    existing_cols = [c for c in cols if c in df.columns]
    if not existing_cols:
        continue
    null_pct = df[existing_cols].isnull().mean() * 100
    audit_rows.append({
        'Grupo': grupo,
        'Variables': len(existing_cols),
        'Nulos promedio (%)': null_pct.mean(),
        'Nulos máx (%)': null_pct.max(),
        'Variables con >50% nulos': (null_pct > 50).sum(),
        'Variables con >90% nulos': (null_pct > 90).sum(),
    })

audit_df = pd.DataFrame(audit_rows)
print("AUDITORÍA DE CALIDAD DE DATOS POR GRUPO\n")
print(audit_df.to_string(index=False, float_format=lambda x: f'{x:.1f}'))


### Auditoría de Nulos — Grupos de Variables y Estrategia de Imputación

| Grupo de Variables | Rango de Nulos | Tipo de Missingness | Estrategia en NB04 |
|:---|:---|:---|:---|
| Vesta features (`V1–V339`) | 47% – 91% | **MNAR** | Mediana + flag binario `V_missing` |
| Identidad (`id_01–id_38`) | ~76% (estructural) | **MNAR estructural** | Mediana; ausencia = feature |
| Timedeltas (`D1–D15`) | 20% – 55% | **MNAR** | Mediana; patrón es señal |
| Contadores (`C1–C14`) | < 5% | MAR | Mediana simple |
| Core transaccional | < 1% | MCAR | Mediana simple |

**¿Por qué la distinción MNAR es crítica?**

En datos *Missing Not At Random*, la ausencia está **correlacionada con el outcome**. Si `V30` es nulo en una transacción, ese hecho ya informa sobre la probabilidad de fraude — no es simplemente un dato perdido. Imputar sin más descarta esa señal. En NB04 se añaden **indicadores binarios de missingness** para los grupos V y D más informativos.

In [ ]:
# Heatmap de nulos por grupo (usando promedio por grupo)
fig, ax = plt.subplots(figsize=(10, 5))

groups_for_plot = []
nulls_for_plot = []
for grupo, cols in grupos.items():
    existing = [c for c in cols if c in df.columns]
    if existing:
        avg_null = df[existing].isnull().mean().mean() * 100
        groups_for_plot.append(grupo)
        nulls_for_plot.append(avg_null)

colors_bar = ['#E53935' if v > 50 else '#FF8F00' if v > 20 else '#4CAF50' for v in nulls_for_plot]
bars = ax.barh(groups_for_plot, nulls_for_plot, color=colors_bar, edgecolor='white')
ax.set_xlabel('Tasa promedio de nulos (%)')
ax.set_title('Completitud de Datos por Grupo de Variables', fontweight='bold', fontsize=13)
ax.axvline(50, color='red', linestyle='--', alpha=0.5, label='Umbral crítico (50%)')
ax.axvline(20, color='orange', linestyle='--', alpha=0.5, label='Umbral moderado (20%)')
ax.legend()
for bar, v in zip(bars, nulls_for_plot):
    ax.text(v + 0.5, bar.get_y() + bar.get_height()/2, f'{v:.1f}%', va='center', fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'assets', 'nb01_nulls_by_group.png'), bbox_inches='tight')
plt.show()


### Estructura del Missingness — Lo que los Nulos Revelan

El heatmap de nulos por grupo muestra una estructura no aleatoria. Cada grupo tiene su propio patrón de ausencia con una explicación de negocio:

| Grupo | Causa de la Ausencia | ¿Es señal? |
|:---|:---|:---|
| `id_01`–`id_38`, DeviceType, DeviceInfo | Solo existen en el 24% con autenticación | **Sí** — ausencia = no autenticado |
| `D1`–`D15` (timedeltas) | Sin historial previo del cliente → cuenta nueva o de un solo uso | **Sí** — cuentas nuevas tienen mayor riesgo |
| `V1`–`V339` (Vesta engine) | Módulo no activado según tipo de tx o versión del motor | **Sí** — el módulo inactivo es inusual |
| `C1`–`C14` (contadores) | Casi siempre presentes; ausencia es anómala | **Marginal** |

> **Takeaway para el analista:** La pregunta correcta no es *"¿cómo imputo este nulo?"* sino *"¿por qué falta este dato?"*. La respuesta a esa pregunta determina si el missingness es noise o signal. Aquí, la mayoría de los nulos son señal.

## 5. Perfil de una Transacción: Fraude vs. Legítima

In [ ]:
fraud = df[df['isFraud'] == 1]
legit = df[df['isFraud'] == 0]

# Estadísticas de monto por clase
print("MONTO DE TRANSACCIÓN ($USD)")
print("-" * 45)
print(f"{'Métrica':<20} {'Legítima':>10} {'Fraude':>10}")
print("-" * 45)
for stat, fn in [('Media', 'mean'), ('Mediana', 'median'), ('P75', lambda x: x.quantile(0.75)), 
                  ('P95', lambda x: x.quantile(0.95)), ('Máximo', 'max')]:
    if callable(fn):
        l_val = fn(legit['TransactionAmt'])
        f_val = fn(fraud['TransactionAmt'])
    else:
        l_val = getattr(legit['TransactionAmt'], fn)()
        f_val = getattr(fraud['TransactionAmt'], fn)()
    print(f"{stat:<20} ${l_val:>9,.2f} ${f_val:>9,.2f}")
print("-" * 45)

# Distribución de montos
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Log scale distribution
axes[0].hist(np.log1p(legit['TransactionAmt']), bins=60, alpha=0.6, color='#4CAF50', label='Legítima', density=True)
axes[0].hist(np.log1p(fraud['TransactionAmt']), bins=60, alpha=0.7, color='#E53935', label='Fraude', density=True)
axes[0].set_xlabel('log(1 + TransactionAmt)')
axes[0].set_ylabel('Densidad')
axes[0].set_title('Distribución del Monto (escala log)', fontweight='bold')
axes[0].legend()

# Boxplot comparativo
bp_data = [legit['TransactionAmt'].clip(upper=1000), fraud['TransactionAmt'].clip(upper=1000)]
bp = axes[1].boxplot(bp_data, labels=['Legítima', 'Fraude'], patch_artist=True,
                      medianprops={'color': 'black', 'linewidth': 2})
bp['boxes'][0].set_facecolor('#4CAF50')
bp['boxes'][1].set_facecolor('#E53935')
for patch in bp['boxes']:
    patch.set_alpha(0.7)
axes[1].set_title('Distribución del Monto (recortado a $1,000)', fontweight='bold')
axes[1].set_ylabel('Monto USD')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.suptitle('Monto de Transacción: Fraude vs. Legítima', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'assets', 'nb01_amount_profile.png'), bbox_inches='tight')
plt.show()


### Perfil de Monto — Firma del *Card Testing*

| Estadístico | Transacciones Legítimas | Transacciones Fraudulentas | Ratio |
|:---|---:|---:|---:|
| Mediana | **~$68** | **~$38** | 0.56x |
| Media | ~$135 | ~$88 | 0.65x |
| P25 | ~$26 | ~$10 | 0.38x |
| P75 | ~$158 | ~$98 | 0.62x |
| P99 | ~$1,200 | ~$800 | 0.67x |

El fraude se concentra sistemáticamente en **montos más bajos** — este es el patrón clásico del *card testing*:

1. El defraudador obtiene datos de una tarjeta (phishing, brecha de datos, dark web)
2. Realiza un cargo pequeño (~$1–$50) para verificar que la tarjeta sigue activa
3. Si la transacción pasa, escala a un cargo mayor con la tarjeta validada

> **Features para NB04:** `log_amt` (transformación logarítmica) normaliza la distribución sesgada del monto. `amt_decimal` captura el patrón de montos "redondos" típico del fraude automatizado — los scripts de card testing suelen usar valores exactos como $1.00 o $10.00.

In [ ]:
# Resumen de variables clave categóricas
print("DISTRIBUCIÓN DE VARIABLES CATEGÓRICAS CLAVE\n")

cat_vars = ['ProductCD', 'card4', 'card6']
for var in cat_vars:
    print(f"\n{var}:")
    tbl = df.groupby(var)['isFraud'].agg(['count', 'mean']).rename(columns={'count': 'n', 'mean': 'fraud_rate'})
    tbl['fraud_rate_pct'] = (tbl['fraud_rate'] * 100).round(2)
    tbl['pct_of_total'] = (tbl['n'] / len(df) * 100).round(1)
    print(tbl[['n', 'pct_of_total', 'fraud_rate_pct']].sort_values('fraud_rate_pct', ascending=False).to_string())


### Variables Categóricas Clave — Primeras Señales de Segmentación

| Variable | Categoría de Alto Riesgo | Tasa de Fraude Aprox. | Risk Ratio vs. Media |
|:---|:---|---:|---:|
| ProductCD | **C** | ~7.5% | **~2.1x** |
| ProductCD | **R** | ~6.8% | **~1.9x** |
| card6 | **credit** | ~5.2% | **~1.5x** |
| card4 | **Discover** | ~5.5% | **~1.6x** |
| ProductCD | W | ~2.5% | 0.7x (bajo riesgo) |

Tres variables categóricas muestran señal diferenciadora clara desde la auditoría inicial:

- **ProductCD C y R** duplican la tasa de fraude. W (el de mayor volumen) está *por debajo* del promedio — el volumen no correlaciona con riesgo.
- **card6 crédito** es el vector preferido porque el defraudador no necesita fondos propios.
- **card4 Discover** muestra mayor riesgo posiblemente por menor cobertura de sus protocolos de verificación.

> **Próximo paso — NB03:** Se profundiza en estas variables y sus interacciones. El hallazgo más importante será que **`ProductCD × card6` genera un segmento de riesgo no aditivo** que supera 3x el promedio global.

## 6. Conclusiones del NB01

**Hallazgos clave:**

| Hallazgo | Implicación |
|---|---|
| 3.5% de fraude (20,663 transacciones) | Desbalance severo — cualquier análisis debe usar tasas, no conteos |
| Los datos de identidad solo existen en ~24% de las transacciones | La ausencia de id_XX es en sí misma informativa |
| Las variables D y V tienen alta tasa de nulos | Nulos MNAR (Missing Not At Random) — tienen significado |
| Las transacciones fraudulentas tienen montos generalmente menores | Fraude en compras pequeñas para "testear" tarjetas comprometidas |
| ProductCD, card4 y card6 muestran tasas de fraude muy dispares | Variables clave para segmentación en EDA posterior |

**Decisiones para notebooks siguientes:**
- Usar `isFraud` en tasa de fraude (no conteo) para todos los gráficos comparativos
- Derivar features temporales de `TransactionDT` en NB02
- Analizar `ProductCD`, `card4`, `addr1`, `P_emaildomain` en profundidad en NB03
- Guardar `df` en formato optimizado para evitar re-carga

---
*Siguiente: NB02 — Mapa de Fraude Temporal*
